# LAMPS D2 Evaluation - New Strategy: RC-PAA

Risk-Calibrated Package Aggregation Agent (RC-PAA) test trên D2 multi-file.

Khác `evaluate_d2_colab.ipynb`:
- Giữ CodeBERT file-level classifier như cũ.
- Giữ baseline VerdictAgent để so sánh.
- Thêm RiskCalibratedVerdictAgent: calibrate score theo critical file, cross-file import, behavior, context penalty.
- Lưu output riêng ở `NT230/data/d2/results_risk_calibrated/`, không overwrite kết quả cũ.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil, sys, json
from pathlib import Path
from collections import defaultdict

DRIVE_D1 = '/content/drive/My Drive/NT230/data/d1'
DRIVE_D2 = '/content/drive/My Drive/NT230/data/d2'
REPO_DIR = Path('/content/NT230')

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
!git clone --depth=1 https://github.com/khoilv2005/NT230.git /content/NT230
sys.path.insert(0, '/content/NT230/src')

# model.bin
os.makedirs('/content/saved_models/checkpoint-best-acc', exist_ok=True)
shutil.copy(f'{DRIVE_D1}/saved_models/checkpoint-best-acc/model.bin',
            '/content/saved_models/checkpoint-best-acc/model.bin')
print('model.bin:', round(os.path.getsize('/content/saved_models/checkpoint-best-acc/model.bin')/1e6), 'MB')

# D2 data
shutil.copy(f'{DRIVE_D2}/files.jsonl',    '/content/d2_files.jsonl')
shutil.copy(f'{DRIVE_D2}/packages.jsonl', '/content/d2_packages.jsonl')
print('d2_files.jsonl:   ', sum(1 for _ in open('/content/d2_files.jsonl')), 'records')
print('d2_packages.jsonl:', sum(1 for _ in open('/content/d2_packages.jsonl')), 'records')

In [ ]:
!pip install -q transformers==4.40.0 torch scikit-learn scipy pandas tqdm

In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU')

In [ ]:
# Imports and agents
from lamps.agents.classifier import ClassifierAgent
from lamps.agents.extractor import ExtractedFile
from lamps.agents.verdict import VerdictAgent
from lamps.agents.risk_calibrated_verdict import RiskCalibratedVerdictAgent
from lamps.evaluation.metrics import classification_report, format_report
from lamps.utils import read_jsonl

classifier = ClassifierAgent(
    checkpoint='/content/saved_models/checkpoint-best-acc/model.bin',
    batch_size=64,
)
baseline_verdict = VerdictAgent(llm=None)
risk_verdict = RiskCalibratedVerdictAgent(
    llm=None,
    package_threshold=0.72,
    critical_bonus=0.18,
    import_bonus=0.14,
    behavior_bonus=0.16,
    low_confidence_penalty=0.12,
    docs_examples_penalty=0.36,
    tooling_penalty=0.24,
    large_package_penalty=0.06,
)
print('Agents ready: CodeBERT + baseline Verdict + RC-PAA Verdict')

In [ ]:
# Load D2
file_records = list(read_jsonl(Path('/content/d2_files.jsonl')))
package_records = list(read_jsonl(Path('/content/d2_packages.jsonl')))
print(f'Files: {len(file_records)} | Packages: {len(package_records)}')

from collections import Counter
print('File labels:', Counter(int(r.get('target', -1)) for r in file_records))
print('Package labels:', Counter(int(r.get('label', -1)) for r in package_records))

In [ ]:
# Extractor Agent - same rule-based filter as evaluate_d2_colab.
NOISY = {'tests', 'test', 'testing', 'docs', 'doc', 'examples', '_vendor', 'vendor'}

def is_relevant(path):
    parts = str(path).lower().replace('\\', '/').split('/')
    return not any(p in NOISY for p in parts) and not parts[-1].startswith('test_')

filtered = [r for r in file_records if is_relevant(r.get('path', ''))]
print(f'After filter: {len(filtered)} / {len(file_records)} files kept')

In [ ]:
# Classifier Agent - CodeBERT per-file with progress
from tqdm.auto import tqdm

files = [
    ExtractedFile(
        package=str(r['package']),
        path=Path('<memory>'),
        rel_path=str(r.get('path', '')),
        source=str(r['func']),
    )
    for r in filtered
]

print(f'Classifying {len(files)} files...')
classifications = []
batch_size = classifier.batch_size
for start in tqdm(range(0, len(files), batch_size), desc='CodeBERT batches'):
    batch = files[start:start + batch_size]
    classifications.extend(classifier.classify_files(batch))

# File-level report is diagnostic only for current D2, because file labels are propagated from package labels.
y_file_true = [int(r['target']) for r in filtered]
y_file_pred = [int(c.target) for c in classifications]
file_report = classification_report(y_file_true, y_file_pred)
print('=== File-level diagnostic / CodeBERT ===')
print(format_report(file_report))

In [ ]:
# Package-level baseline vs RC-PAA
cls_by_pkg = defaultdict(list)
files_by_pkg = defaultdict(list)
for cls, extracted in zip(classifications, files):
    cls_by_pkg[cls.package].append(cls)
    files_by_pkg[extracted.package].append(extracted)

y_true = []
y_baseline = []
y_rcpaa = []
package_results = []
risk_rows = []

for pkg in package_records:
    name = pkg['package']
    true_label = int(pkg['label'])
    preds = cls_by_pkg.get(name, [])
    pkg_files = files_by_pkg.get(name, [])

    base_v = baseline_verdict.aggregate(name, preds)
    rc_v = risk_verdict.aggregate(name, preds, pkg_files)
    top_risks = sorted(risk_verdict.last_file_risks, key=lambda r: r.calibrated_score, reverse=True)

    y_true.append(true_label)
    y_baseline.append(int(base_v.target))
    y_rcpaa.append(int(rc_v.target))

    package_results.append({
        'package': name,
        'target': true_label,
        'baseline_predicted': int(base_v.target),
        'rcpaa_predicted': int(rc_v.target),
        'n_files_total': int(pkg.get('n_files', 0)),
        'n_files_after_filter': len(preds),
        'baseline_malicious_files': len(base_v.malicious_files),
        'rcpaa_malicious_files': len(rc_v.malicious_files),
        'max_base_score': max([p.score for p in preds], default=0.0),
        'max_calibrated_score': max([r.calibrated_score for r in top_risks], default=0.0),
        'top_risk_files': [
            {
                'path': r.rel_path,
                'base_score': round(r.base_score, 4),
                'calibrated_score': round(r.calibrated_score, 4),
                'reasons': r.reasons,
            }
            for r in top_risks[:5]
        ],
        'rationale': rc_v.rationale,
    })

    for r in risk_verdict.risk_rows():
        risk_rows.append(r)

baseline_report = classification_report(y_true, y_baseline)
rcpaa_report = classification_report(y_true, y_rcpaa)

print('=== Package-level baseline conservative Verdict ===')
print(format_report(baseline_report))
print('\n=== Package-level RC-PAA new strategy ===')
print(format_report(rcpaa_report))

In [ ]:
# Threshold diagnostic for RC-PAA without re-running CodeBERT
threshold_rows = []
for th in [0.50, 0.55, 0.60, 0.65, 0.70, 0.72, 0.75, 0.80, 0.85, 0.90]:
    preds = [1 if row['max_calibrated_score'] >= th else 0 for row in package_results]
    rep = classification_report(y_true, preds)
    d = rep.to_dict()
    threshold_rows.append({
        'threshold': th,
        'accuracy': d['accuracy'],
        'balanced_accuracy': d['balanced_accuracy'],
        'precision': d['precision'],
        'recall': d['recall'],
        'f1': d['f1'],
        'TN': d['confusion']['TN'],
        'FP': d['confusion']['FP'],
        'FN': d['confusion']['FN'],
        'TP': d['confusion']['TP'],
    })

print('threshold, accuracy, balanced_accuracy, precision, recall, f1, TN, FP, FN, TP')
for r in threshold_rows:
    print(f"{r['threshold']:.2f}, {r['accuracy']:.4f}, {r['balanced_accuracy']:.4f}, {r['precision']:.4f}, {r['recall']:.4f}, {r['f1']:.4f}, {r['TN']}, {r['FP']}, {r['FN']}, {r['TP']}")

In [ ]:
# Export wrong predictions and comparison artifacts
import csv

OUTPUT_DIR = Path('/content/results_d2_risk_calibrated')
DRIVE_OUTPUT_DIR = Path(DRIVE_D2) / 'results_risk_calibrated'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def write_jsonl(path, rows):
    path.write_text(
        '\n'.join(json.dumps(r, ensure_ascii=False) for r in rows) + ('\n' if rows else ''),
        encoding='utf-8',
    )

def write_csv(path, rows, fieldnames):
    with path.open('w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction='ignore')
        writer.writeheader()
        writer.writerows(rows)

wrong_rcpaa = []
changed = []
for row in package_results:
    row = {**row}
    if int(row['target']) != int(row['rcpaa_predicted']):
        row['error_type'] = 'FP' if int(row['target']) == 0 else 'FN'
        wrong_rcpaa.append(row)
    if int(row['baseline_predicted']) != int(row['rcpaa_predicted']):
        changed.append(row)

wrong_rcpaa_sorted = sorted(wrong_rcpaa, key=lambda r: (r['error_type'], -float(r['max_calibrated_score']), r['package']))
changed_sorted = sorted(changed, key=lambda r: (-float(r['max_calibrated_score']), r['package']))

for out_dir in (OUTPUT_DIR, DRIVE_OUTPUT_DIR):
    (out_dir / 'file_report.json').write_text(json.dumps(file_report.to_dict(), indent=2), encoding='utf-8')
    (out_dir / 'baseline_package_report.json').write_text(json.dumps(baseline_report.to_dict(), indent=2), encoding='utf-8')
    (out_dir / 'rcpaa_package_report.json').write_text(json.dumps(rcpaa_report.to_dict(), indent=2), encoding='utf-8')
    (out_dir / 'baseline_package_report.txt').write_text(format_report(baseline_report), encoding='utf-8')
    (out_dir / 'rcpaa_package_report.txt').write_text(format_report(rcpaa_report), encoding='utf-8')
    write_jsonl(out_dir / 'package_predictions.jsonl', package_results)
    write_jsonl(out_dir / 'risk_file_scores.jsonl', risk_rows)
    write_jsonl(out_dir / 'rcpaa_wrong_predictions.jsonl', wrong_rcpaa_sorted)
    write_jsonl(out_dir / 'rcpaa_changed_vs_baseline.jsonl', changed_sorted)
    write_jsonl(out_dir / 'threshold_diagnostic.jsonl', threshold_rows)
    write_csv(
        out_dir / 'rcpaa_wrong_predictions.csv',
        wrong_rcpaa_sorted,
        ['package', 'error_type', 'target', 'baseline_predicted', 'rcpaa_predicted', 'n_files_total',
         'n_files_after_filter', 'baseline_malicious_files', 'rcpaa_malicious_files',
         'max_base_score', 'max_calibrated_score', 'top_risk_files', 'rationale'],
    )
    write_csv(
        out_dir / 'rcpaa_changed_vs_baseline.csv',
        changed_sorted,
        ['package', 'target', 'baseline_predicted', 'rcpaa_predicted', 'n_files_total',
         'n_files_after_filter', 'baseline_malicious_files', 'rcpaa_malicious_files',
         'max_base_score', 'max_calibrated_score', 'top_risk_files', 'rationale'],
    )
    (out_dir / 'summary.json').write_text(json.dumps({
        'n_packages': len(package_results),
        'n_filtered_files': len(files),
        'output': str(out_dir),
        'baseline_package_report': baseline_report.to_dict(),
        'rcpaa_package_report': rcpaa_report.to_dict(),
        'rcpaa_wrong': len(wrong_rcpaa_sorted),
        'changed_vs_baseline': len(changed_sorted),
        'threshold': risk_verdict.package_threshold,
    }, indent=2), encoding='utf-8')

print('Saved to Drive:', DRIVE_OUTPUT_DIR)
print('rcpaa_package_report.json')
print('rcpaa_wrong_predictions.csv')
print('rcpaa_changed_vs_baseline.csv')
print('risk_file_scores.jsonl')